# HW5 第一题：CV球粒陨石氧同位素混合计算

## 已知条件

- CV球粒陨石：δ¹⁸O = 0‰, δ¹⁷O = -4‰ (Wasson, 1974)
- 假设CV球粒陨石中的氧由两部分混合而成：
  1. 位于地球氧同位素质量分馏线（Terrestrial Fractionation Line, TFL）上的成分
  2. 纯¹⁶O端元物质

地球质量分馏线（TFL）：δ¹⁷O = λ × δ¹⁸O，其中 λ ≈ 0.52

纯¹⁶O端元：只含¹⁶O，不含¹⁷O或¹⁸O，因此 R₁₇ = 0, R₁₈ = 0，即 δ¹⁷O = -1000‰, δ¹⁸O = -1000‰

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

## 1. 数学推导

### 定义

设 R₁₈ = ¹⁸O/¹⁶O（相对SMOW归一化），R₁₇ = ¹⁷O/¹⁶O（相对SMOW归一化）

δ值与R值的关系：
$$R = \frac{\delta}{1000} + 1$$

### 二元混合

设 TFL 端元占比为 $f$，纯¹⁶O端元占比为 $1-f$。混合后各同位素比值满足：

$$R_{18}^{CV} = f \cdot R_{18}^{TFL} + (1-f) \cdot 0$$
$$R_{17}^{CV} = f \cdot R_{17}^{TFL} + (1-f) \cdot 0$$

因此：
$$\frac{R_{17}^{CV}}{R_{18}^{CV}} = \frac{R_{17}^{TFL}}{R_{18}^{TFL}}$$

即混合线上所有点的 R₁₇/R₁₈ 比值恒定。

### 求解TFL端元组成

TFL约束：$\delta^{17}O_{TFL} = \lambda \cdot \delta^{18}O_{TFL}$

即：$R_{17}^{TFL} - 1 = \lambda (R_{18}^{TFL} - 1)$，或 $R_{17}^{TFL} = \lambda R_{18}^{TFL} + (1-\lambda)$

混合约束：$R_{17}^{TFL} = \frac{R_{17}^{CV}}{R_{18}^{CV}} \cdot R_{18}^{TFL}$

联立求解即可得到 TFL 端元的组成，进而求得 $f$。

In [ ]:
# ========== 已知参数 ==========
d18O_CV = 0.0      # CV球粒陨石 δ¹⁸O (‰)
d17O_CV = -4.0     # CV球粒陨石 δ¹⁷O (‰)
lam = 0.52         # 地球质量分馏线斜率

# 纯¹⁶O端元
d18O_16O = -1000.0  # ‰
d17O_16O = -1000.0  # ‰

# 将δ值转换为同位素比值（相对于SMOW归一化）
R18_CV = d18O_CV / 1000.0 + 1.0    # = 1.0
R17_CV = d17O_CV / 1000.0 + 1.0    # = 0.996

# 纯¹⁶O的比值
R18_16 = 0.0
R17_16 = 0.0

print(f"CV球粒陨石: R₁₈ = {R18_CV:.6f}, R₁₇ = {R17_CV:.6f}")
print(f"R₁₇/R₁₈ = {R17_CV/R18_CV:.6f}")

In [ ]:
# ========== 求解TFL端元组成 ==========
# 从 TFL 约束: R17_TFL = lam * R18_TFL + (1 - lam)
# 从混合约束: R17_TFL = (R17_CV / R18_CV) * R18_TFL
# 联立: (R17_CV / R18_CV) * R18_TFL = lam * R18_TFL + (1 - lam)

ratio_CV = R17_CV / R18_CV  # 混合线的 R₁₇/R₁₈

# 求解 R18_TFL
R18_TFL = (1.0 - lam) / (ratio_CV - lam)
R17_TFL = ratio_CV * R18_TFL

# 转换为δ值
d18O_TFL = (R18_TFL - 1.0) * 1000.0
d17O_TFL = (R17_TFL - 1.0) * 1000.0

print("=" * 50)
print("TFL端元组成 (混合线与TFL的交点):")
print(f"  R₁₈_TFL = {R18_TFL:.6f}")
print(f"  R₁₇_TFL = {R17_TFL:.6f}")
print(f"  δ¹⁸O_TFL = {d18O_TFL:.2f}‰")
print(f"  δ¹⁷O_TFL = {d17O_TFL:.2f}‰")
print(f"  验证TFL: δ¹⁷O/δ¹⁸O = {d17O_TFL/d18O_TFL:.6f}")

In [ ]:
# ========== 计算纯¹⁶O的加入量 ==========
# 从 R18 混合方程: R18_CV = f * R18_TFL + (1-f) * 0
f_TFL = R18_CV / R18_TFL
f_16O = 1.0 - f_TFL

print("=" * 50)
print("混合比例:")
print(f"  TFL端元占比 f   = {f_TFL:.6f} ({f_TFL*100:.4f}%)")
print(f"  纯¹⁶O占比 1-f  = {f_16O:.6f} ({f_16O*100:.4f}%)")

# 验证：用δ空间近似验算
d18O_check = f_TFL * d18O_TFL + f_16O * d18O_16O
d17O_check = f_TFL * d17O_TFL + f_16O * d17O_16O
print(f"\n验证 (δ空间混合):")
print(f"  δ¹⁸O_check = {d18O_check:.4f}‰ (目标 0‰)")
print(f"  δ¹⁷O_check = {d17O_check:.4f}‰ (目标 -4‰)")

# 用R空间精确验算
R18_check = f_TFL * R18_TFL + f_16O * R18_16
R17_check = f_TFL * R17_TFL + f_16O * R17_16
print(f"\n验证 (R空间混合):")
print(f"  R₁₈_check = {R18_check:.6f} (目标 {R18_CV:.6f})")
print(f"  R₁₇_check = {R17_check:.6f} (目标 {R17_CV:.6f})")

## 2. 混合示意图

下图中：
- **蓝色实线**：地球质量分馏线 (TFL, δ¹⁷O = 0.52 × δ¹⁸O)
- **红色虚线**：¹⁶O混合线（纯¹⁶O端元 → TFL端元）
- **绿色星形**：CV球粒陨石 (0‰, -4‰)
- **橙色方形**：TFL端元（混合线与TFL的交点）
- **蓝色圆形**：纯¹⁶O端元 (-1000‰, -1000‰)

In [ ]:
# ========== 生成绘图数据 ==========

# TFL线：从原点附近到TFL端元以上
d18O_TFL_line = np.linspace(-10, d18O_TFL + 5, 100)
d17O_TFL_line = lam * d18O_TFL_line

# 混合线：从纯¹⁶O到TFL端元
# 混合线方程为 δ¹⁷O = -1000 + (1000 + d17O_TFL) / (1000 + d18O_TFL) * (δ¹⁸O + 1000)
slope_mix = (1000 + d17O_TFL) / (1000 + d18O_TFL)
d18O_mix_line = np.linspace(-1000, d18O_TFL, 200)
# 用通过两点(-1000,-1000)和(d18O_TFL, d17O_TFL)的直线
d17O_mix_line = -1000 + (d17O_TFL + 1000) / (d18O_TFL + 1000) * (d18O_mix_line + 1000)

# 地球水 (SMOW) 原点
d18O_SMOW = 0.0
d17O_SMOW = 0.0

In [ ]:
# ========== 主图: 完整混合示意图 ==========
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# TFL线
ax.plot(d18O_TFL_line, d17O_TFL_line, 'b-', linewidth=2, label='TFL ($\delta^{17}$O = 0.52 $\delta^{18}$O)')

# 混合线（只画到可视范围内，避免坐标轴被拉得过远）
d18O_visible = np.linspace(-50, d18O_TFL, 200)
d17O_visible = -1000 + (d17O_TFL + 1000) / (d18O_TFL + 1000) * (d18O_visible + 1000)
ax.plot(d18O_visible, d17O_visible, 'r--', linewidth=2, label='$^{16}$O mixing line')

# CV球粒陨石
ax.plot(d18O_CV, d17O_CV, 'g*', markersize=16, markeredgewidth=1.5,
        markeredgecolor='black', zorder=5, label=f'CV chondrite ({d18O_CV}‰, {d17O_CV}‰)')

# TFL端元
ax.plot(d18O_TFL, d17O_TFL, 's', color='orange', markersize=12, markeredgewidth=1.5,
        markeredgecolor='black', zorder=5, label=f'TFL endmember ({d18O_TFL:.1f}‰, {d17O_TFL:.1f}‰)')

# SMOW原点
ax.plot(d18O_SMOW, d17O_SMOW, '^', color='gray', markersize=10, label='SMOW (0, 0)')

# 用箭头指向纯¹⁶O端元（不在可视范围内）
ax.annotate('Pure $^{16}$O\n($-$1000‰, $-$1000‰)',
            xy=(-35, -40), xytext=(-45, -48),
            fontsize=9, color='blue', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))

# 坐标轴
ax.axhline(y=0, color='gray', linewidth=0.5, linestyle=':')
ax.axvline(x=0, color='gray', linewidth=0.5, linestyle=':')

ax.set_xlabel('$\delta^{18}$O (‰, SMOW)')
ax.set_ylabel('$\delta^{17}$O (‰, SMOW)')
ax.set_title('CV chondrite oxygen isotope mixing diagram')
ax.legend(loc='upper left', fontsize=9)
ax.set_aspect('equal')

# 手动设置坐标范围，使TFL和CV区域清晰可见
ax.set_xlim(-50, d18O_TFL + 10)
ax.set_ylim(-55, d17O_TFL + 10)

# 标注混合比例
ax.annotate(f'Pure $^{16}$O fraction\n1$-$f = {f_16O*100:.3f}%',
            xy=(d18O_CV, d17O_CV), xytext=(d18O_CV - 30, d17O_CV + 8),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig1_mixing_diagram.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ========== 放大图: 展示CV附近区域 ==========
fig, ax = plt.subplots(1, 1, figsize=(8, 7))

# TFL线
ax.plot(d18O_TFL_line, d17O_TFL_line, 'b-', linewidth=2, label='TFL ($\delta^{17}$O = 0.52 $\delta^{18}$O)')

# 混合线
ax.plot(d18O_mix_line, d17O_mix_line, 'r--', linewidth=2, label='$^{16}$O mixing line')

# CV球粒陨石
ax.plot(d18O_CV, d17O_CV, 'g*', markersize=16, markeredgewidth=2,
        markeredgecolor='black', zorder=5, label=f'CV chondrite ({d18O_CV}‰, {d17O_CV}‰)')

# TFL端元
ax.plot(d18O_TFL, d17O_TFL, 's', color='orange', markersize=12, markeredgewidth=2,
        markeredgecolor='black', zorder=5, label=f'TFL endmember ({d18O_TFL:.1f}‰, {d17O_TFL:.1f}‰)')

# SMOW
ax.plot(d18O_SMOW, d17O_SMOW, '^', color='gray', markersize=10, label='SMOW (0, 0)')

# 坐标轴
ax.axhline(y=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

# 设置放大范围
margin = 2
ax.set_xlim(d18O_TFL + margin, -margin)
ax.set_ylim(min(d17O_CV, 0) - margin, max(d17O_TFL, 0) + margin)

ax.set_xlabel('$\delta^{18}$O (‰, SMOW)')
ax.set_ylabel('$\delta^{17}$O (‰, SMOW)')
ax.set_title('Zoom-in: CV chondrite and TFL endmember region')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('figures/fig2_zoom_in.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. 结论

CV球粒陨石的氧同位素组成可以通过以下二元混合获得：

- **TFL端元**：位于地球质量分馏线上，δ¹⁸O ≈ 8.40‰, δ¹⁷O ≈ 4.37‰
- **纯¹⁶O端元**：δ¹⁸O = -1000‰, δ¹⁷O = -1000‰

需要加入的纯¹⁶O端元物质的**质量分数约为 0.833%**（即约 0.83 wt% 的纯¹⁶O与 99.17 wt% 的TFL端元混合）。

该结果显示CV球粒陨石中含有非常少量的¹⁶O-rich组分，这与CAIs（富钙铝包体）中含有显著的¹⁶O过剩现象一致，反映了早期太阳系中¹⁶O-rich储库的存在。在δ¹⁷O-δ¹⁸O图上，CV球粒陨石位于TFL下方，其混合线斜率（≈0.996）明显大于TFL斜率（0.52），这也支持了碳质球粒陨石中 anhydrous minerals 的CCAM（Carbonaceous Chondrite Anhydrous Mineral）混合线的经典解释。